
# Week 6 – Making the Baseline CNN Trustworthy & UI-Ready

**Important:**  
This notebook does **NOT** create a new CNN from scratch.

It **loads the Week 5 baseline CNN**, improves its behavior slightly,  
adds **confidence handling and explainability**, and saves the **same model**  
that will be used by the **Gradio UI in Week 7**.



## Step 1 – Load Baseline CNN (from Week 5)


In [ ]:

import tensorflow as tf

# Path where Week 5 model was saved
MODEL_PATH = "baseline_cnn_xray.keras"

model = tf.keras.models.load_model(MODEL_PATH)
model.summary()



## Step 2 – Evaluate Baseline Model (Reality Check)


In [ ]:

test_loss, test_acc, test_auc = model.evaluate(test_ds)

print("Accuracy:", test_acc)
print("AUC:", test_auc)



## Step 3 – Light Improvement (Stability, Not New Architecture)
We **do NOT** build a new CNN.
We only:
- Add dropout if missing
- Recompile
- Retrain briefly


In [ ]:

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)



## Step 4 – Understanding Predictions (Confidence & Thresholds)


In [ ]:

import numpy as np
from sklearn.metrics import confusion_matrix

y_true, y_probs = [], []

for images, labels in val_ds:
    probs = model.predict(images)
    y_probs.extend(probs.flatten())
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_probs = np.array(y_probs)

def evaluate_threshold(threshold):
    preds = (y_probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    return {
        "threshold": threshold,
        "sensitivity": tp / (tp + fn),
        "specificity": tn / (tn + fp)
    }

for t in [0.3, 0.5, 0.7]:
    print(evaluate_threshold(t))



## Step 5 – Severity Bands (What UI Will Show)


In [ ]:

def severity_band(prob):
    if prob < 0.3:
        return "LOW"
    elif prob < 0.7:
        return "UNCERTAIN"
    else:
        return "HIGH"

sample_probs = y_probs[:10]
for p in sample_probs:
    print(p, "→", severity_band(p))



## Step 6 – Explainability with Grad-CAM (Minimal)


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        model.inputs,
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        loss = preds[:, 0]

    grads = tape.gradient(loss, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))

    heatmap = tf.reduce_sum(pooled_grads * conv_out[0], axis=-1)
    heatmap = tf.maximum(heatmap, 0) / tf.reduce_max(heatmap)
    return heatmap.numpy()



## Step 7 – Save FINAL MODEL (Used by Gradio UI)


In [ ]:

FINAL_MODEL_PATH = "final_cnn_xray_ui_ready.keras"
model.save(FINAL_MODEL_PATH)

print("Final model saved for Week 7 UI:", FINAL_MODEL_PATH)
